# 📰 Fake News Detection with BERT
**Model:** `jy46604790/Fake-News-Bert-Detect` from HuggingFace

This notebook loads a pretrained BERT model fine-tuned for fake news detection and lets you run predictions on custom text.

## 1. Install Dependencies

In [ ]:
!pip install transformers torch -q

## 2. Imports

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import pandas as pd
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 3. Load Model & Tokenizer from HuggingFace

> The model will be downloaded automatically. Labels: **0 = Real**, **1 = Fake**

In [ ]:
MODEL_NAME = 'jy46604790/Fake-News-Bert-Detect'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print('Loading model...')
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print('✅ Model loaded successfully!')
print(f'Labels: {model.config.id2label}')

## 4. Prediction Function

In [ ]:
def predict(text, max_length=512):
    """
    Predict whether a news article is REAL or FAKE.
    
    Args:
        text (str): The news article or headline text.
        max_length (int): Max token length for BERT (default 512).
    
    Returns:
        dict: label, confidence, and probabilities for both classes.
    """
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
        padding='max_length'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
    pred_id = int(np.argmax(probs))
    label = model.config.id2label[pred_id]
    confidence = float(probs[pred_id])

    return {
        'label': label,
        'confidence': round(confidence * 100, 2),
        'prob_real': round(float(probs[0]) * 100, 2),
        'prob_fake': round(float(probs[1]) * 100, 2)
    }


def predict_batch(texts, batch_size=16, max_length=512):
    """
    Predict a list of texts in batches.
    
    Args:
        texts (list[str]): List of news texts.
        batch_size (int): Number of samples per batch.
        max_length (int): Max token length.
    
    Returns:
        list[dict]: Predictions for each text.
    """
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            truncation=True,
            max_length=max_length,
            padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        for prob in probs:
            pred_id = int(np.argmax(prob))
            results.append({
                'label': model.config.id2label[pred_id],
                'confidence': round(float(prob[pred_id]) * 100, 2),
                'prob_real': round(float(prob[0]) * 100, 2),
                'prob_fake': round(float(prob[1]) * 100, 2)
            })
    return results

## 5. Single Text Prediction

In [ ]:
# ✏️ Change this text to test your own news
sample_text = """
Scientists have confirmed that drinking coffee every morning significantly extends
human lifespan by up to 20 years, according to a new study published in the
Journal of Medical Science.
"""

result = predict(sample_text)

print(f"📋 Prediction : {result['label']}")
print(f"🎯 Confidence : {result['confidence']}%")
print(f"✅ Prob Real  : {result['prob_real']}%")
print(f"❌ Prob Fake  : {result['prob_fake']}%")

## 6. Batch Prediction on Multiple Articles

In [ ]:
# ✏️ Add your own list of news articles or headlines
articles = [
    "NASA confirms first human landing on Mars scheduled for 2025.",
    "The stock market closed higher on Friday amid positive economic data.",
    "Government secretly replaced tap water with mind-control chemicals.",
    "WHO declares new global health emergency over rising flu cases.",
    "Aliens have contacted world leaders and the truth is being hidden from us."
]

batch_results = predict_batch(articles)

df = pd.DataFrame({
    'text': [t[:80] + '...' if len(t) > 80 else t for t in articles],
    'label': [r['label'] for r in batch_results],
    'confidence (%)': [r['confidence'] for r in batch_results],
    'prob_real (%)': [r['prob_real'] for r in batch_results],
    'prob_fake (%)': [r['prob_fake'] for r in batch_results]
})

df

## 7. (Optional) Run on a CSV Dataset

If you have a CSV file with a `text` column (e.g., from a Kaggle dataset), use this cell.

In [ ]:
# ✏️ Set your CSV path and text column name
CSV_PATH = '/kaggle/input/your-dataset/news.csv'   # <-- change this
TEXT_COLUMN = 'text'                                # <-- change if needed

# Uncomment to run:
# df_data = pd.read_csv(CSV_PATH)
# texts = df_data[TEXT_COLUMN].fillna('').tolist()
#
# print(f'Running inference on {len(texts)} samples...')
# preds = predict_batch(texts, batch_size=32)
#
# df_data['predicted_label'] = [p['label'] for p in preds]
# df_data['confidence'] = [p['confidence'] for p in preds]
# df_data['prob_fake'] = [p['prob_fake'] for p in preds]
#
# print(df_data[['text', 'predicted_label', 'confidence']].head(10))
# df_data.to_csv('predictions.csv', index=False)
# print('✅ Saved to predictions.csv')

## 8. (Optional) Evaluate Against Ground Truth Labels

In [ ]:
# Uncomment after running the CSV section above and if your CSV has a 'label' column

# from sklearn.metrics import classification_report, confusion_matrix
# import seaborn as sns
# import matplotlib.pyplot as plt
#
# LABEL_COLUMN = 'label'   # <-- your ground truth column
#
# y_true = df_data[LABEL_COLUMN].tolist()
# y_pred = df_data['predicted_label'].tolist()
#
# print(classification_report(y_true, y_pred))
#
# cm = confusion_matrix(y_true, y_pred)
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#             xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
# plt.title('Confusion Matrix')
# plt.ylabel('Actual')
# plt.xlabel('Predicted')
# plt.show()